In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

In [ ]:
from datasets import load_dataset

ds = load_dataset("seeeeiii/RICO-WidgetCaptioning")

In [ ]:
ds

In [ ]:
train_ds = ds['train']
valid_ds = ds['val']
test_ds = ds['test']

In [ ]:
del ds

In [ ]:
def simplify_ds(ds):
  compact = ds.select_columns(
      ["screenId", "captions", "category", "app_package_name"]
  ).rename_columns({
      "screenId": "id",
      "app_package_name": "package_name",
  })
  return compact

In [ ]:
categories = train_ds.unique("category")

print("Number of categories:", len(categories))
print(sorted(categories))

In [ ]:
train_ds = simplify_ds(train_ds)
valid_ds = simplify_ds(valid_ds)
test_ds = simplify_ds(test_ds)

In [ ]:
def split_ds(ds):
  return ds.map(
    lambda row: {
      "text": "|".join(row['captions']),
      "label": row['category']
    }
  )

In [ ]:
train_ds = split_ds(train_ds)
valid_ds = split_ds(valid_ds)
test_ds = split_ds(test_ds)
train_ds

In [ ]:
X_train, y_train = train_ds["text"], train_ds["label"]
X_valid, y_valid = valid_ds["text"], valid_ds["label"]
X_test, y_test = test_ds["text"], test_ds["label"]


In [ ]:
len(X_train), len(y_train)

In [ ]:
X_train[0], y_train[0]

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_valid_tfidf = vectorizer.transform(X_valid)

In [ ]:
print(X_train_tfidf.shape)
print(vectorizer.get_feature_names_out()[:20])

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

In [ ]:
model.fit(X_train_tfidf, y_train)

valid_predictions = model.predict(X_valid_tfidf)

print("Validation accuracy:", accuracy_score(y_valid, valid_predictions))
print(classification_report(y_valid, valid_predictions))

In [ ]:
for text, actual, predicted in zip(X_valid[:10], y_valid[:10], valid_predictions[:10]):
    print(f"Text:      {text}")
    print(f"Actual:    {actual}")
    print(f"Predicted: {predicted}")
    print()

In [ ]:
from collections import Counter

train_counts = Counter(y_train)

for category, count in train_counts.most_common():
    print(f"{category}: {count}")

In [ ]:
most_common_category, count = train_counts.most_common(1)[0]
baseline_accuracy = count / len(y_train)

print(most_common_category)
print(f"Baseline accuracy: {baseline_accuracy:.2%}")

In [ ]:
print(set(y_valid) - set(y_train))

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
    )),
])

In [ ]:
pipeline.fit(X_train, y_train)

valid_predictions = pipeline.predict(X_valid)

print(f"Validation accuracy: {accuracy_score(y_valid, valid_predictions):.2%}")
print(classification_report(y_valid, valid_predictions))

In [ ]:
for text, actual, predicted in zip(X_valid[:10], y_valid[:10], valid_predictions[:10]):
    print(f"Caption:   {text}")
    print(f"Actual:    {actual}")
    print(f"Predicted: {predicted}\n")

In [ ]:
from pathlib import Path
import json

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType

export_dir = Path("models")
export_dir.mkdir(exist_ok=True)

tfidf_options = {
    "tfidf": {
        "separators": [
            " ", "[.]", "\\?", ",", ";", ":", "\\!",
            "\\(", "\\)", "\\[", "\\]", "\\|", "\\-", "\n",
            "\"", "'",
        ]
    },
    "classifier": {
        "zipmap": False,
        "nocl": True,
    },
}

onnx_model = convert_sklearn(
    pipeline,
    initial_types=[("text", StringTensorType([None, 1]))],
    options=tfidf_options,
    target_opset=17,
)

onnx_path = export_dir / "category_classifier.onnx"
onnx_path.write_bytes(onnx_model.SerializeToString())

labels_path = export_dir / "labels.json"
labels_path.write_text(
    json.dumps(pipeline.named_steps["classifier"].classes_.tolist()),
    encoding="utf-8",
)

print(onnx_path.resolve())
print(labels_path.resolve())

In [ ]:
import numpy as np
import onnxruntime as ort

sample_texts = X_valid[:10]

session = ort.InferenceSession(
    "models/category_classifier.onnx",
    providers=["CPUExecutionProvider"],
)

onnx_inputs = np.asarray(sample_texts, dtype=object).reshape(-1, 1)
outputs = session.run(None, {"text": onnx_inputs})

print([output.shape for output in outputs if hasattr(output, "shape")])
print("Python:", pipeline.predict(sample_texts))

In [ ]:
print([item.name for item in session.get_inputs()])
print([item.name for item in session.get_outputs()])